In [1]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score


In [4]:
ANNOT = Path("linkedin-cvs-annotated.json")
UNLAB = Path("linkedin-cvs-not-annotated.json")

assert ANNOT.exists(), f"Missing file: {ANNOT}"
assert UNLAB.exists(), f"Missing file: {UNLAB}"

with open(ANNOT, "r", encoding="utf-8") as f:
    profiles_annot = json.load(f)

with open(UNLAB, "r", encoding="utf-8") as f:
    profiles_unlab = json.load(f)

# flatten list-of-lists if needed
if isinstance(profiles_annot, list) and len(profiles_annot) > 0 and isinstance(profiles_annot[0], list):
    profiles_annot = [x for sub in profiles_annot for x in sub]

if isinstance(profiles_unlab, list) and len(profiles_unlab) > 0 and isinstance(profiles_unlab[0], list):
    profiles_unlab = [x for sub in profiles_unlab for x in sub]

print("Annotated profiles:", len(profiles_annot))
print("Unlabeled profiles:", len(profiles_unlab))
print("Example keys:", list(profiles_annot[0].keys())[:25])


Annotated profiles: 2638
Unlabeled profiles: 1886
Example keys: ['organization', 'linkedin', 'position', 'startDate', 'endDate', 'status', 'department', 'seniority']


In [5]:
def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, dict):
        x = " ".join([str(v) for v in x.values() if v is not None])
    elif isinstance(x, (list, tuple)):
        x = " ".join([str(v) for v in x if v is not None])
    else:
        x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def safe_get(d, keys, default=None):
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

def build_text(profile):
    pos   = clean_text(safe_get(profile, ["position", "title", "jobTitle"], ""))
    org   = clean_text(safe_get(profile, ["organization", "company", "employer"], ""))
    stat  = clean_text(safe_get(profile, ["status"], ""))
    sd    = clean_text(safe_get(profile, ["startDate", "start_date"], ""))
    ed    = clean_text(safe_get(profile, ["endDate", "end_date"], ""))

    headline = clean_text(safe_get(profile, ["headline"], ""))
    summary  = clean_text(safe_get(profile, ["summary", "about"], ""))
    skills   = clean_text(safe_get(profile, ["skills", "skill"], ""))
    desc     = clean_text(safe_get(profile, ["description", "job_description", "jobDescription"], ""))

    parts = [
        f"position: {pos}" if pos else "",
        f"org: {org}" if org else "",
        f"headline: {headline}" if headline else "",
        f"summary: {summary}" if summary else "",
        f"skills: {skills}" if skills else "",
        f"desc: {desc}" if desc else "",
        f"start: {sd}" if sd else "",
        f"end: {ed}" if ed else "",
        f"status: {stat}" if stat else "",
    ]
    parts = [p for p in parts if p]
    return " | ".join(parts)


In [6]:
def get_labels(profile):
    dom = safe_get(profile, ["department", "domain"], None)
    sen = safe_get(profile, ["seniority", "seniority_level"], None)

    labels = safe_get(profile, ["labels", "annotation", "annotations"], None)
    if isinstance(labels, dict):
        dom = dom or safe_get(labels, ["department", "domain"], None)
        sen = sen or safe_get(labels, ["seniority", "seniority_level"], None)

    return dom, sen

rows = []
for i, p in enumerate(profiles_annot):
    dom, sen = get_labels(p)
    rows.append({
        "idx": i,
        "text": build_text(p),
        "domain_true": clean_text(dom) if dom is not None else None,
        "seniority_true": clean_text(sen) if sen is not None else None,
        "status": clean_text(safe_get(p, ["status"], "")),
    })

df_gold = pd.DataFrame(rows)

df_gold = df_gold[
    (df_gold["text"].str.len() > 0) &
    (df_gold["domain_true"].notna()) &
    (df_gold["seniority_true"].notna())
].copy()

print("Gold usable rows:", len(df_gold))
print("Domains:", df_gold["domain_true"].nunique())
print("Seniorities:", df_gold["seniority_true"].nunique())
df_gold.head()


Gold usable rows: 2638
Domains: 11
Seniorities: 6


,idx,text,domain_true,seniority_true,status
0,0,position: Prokurist | org: Depot4Design GmbH |...,Other,Management,ACTIVE
1,1,position: CFO | org: Depot4Design GmbH | start...,Other,Management,ACTIVE
2,2,position: Betriebswirtin | org: Depot4Design G...,Other,Professional,ACTIVE
3,3,position: Prokuristin | org: Depot4Design GmbH...,Other,Management,ACTIVE
4,4,position: CFO | org: Depot4Design GmbH | start...,Other,Management,ACTIVE


In [7]:
rows_un = []
for i, p in enumerate(profiles_unlab):
    rows_un.append({
        "idx": i,
        "text": build_text(p),
        "status": clean_text(safe_get(p, ["status"], "")),
    })

df_un = pd.DataFrame(rows_un)
df_un = df_un[df_un["text"].str.len() > 0].copy()

print("Unlabeled usable rows:", len(df_un))
df_un.head()


Unlabeled usable rows: 1886


,idx,text,status
0,0,"position: Bookkeeper | org: Keeping The Books,...",ACTIVE
1,1,position: Co-Owner | org: Playful Paws | start...,ACTIVE
2,2,position: Logistics Officer | org: S&R service...,INACTIVE
3,3,position: Truck driver/ laborer | org: ABC Sup...,INACTIVE
4,4,position: Fuel Driver | org: MB Railways | sta...,INACTIVE


In [8]:
DOMAIN_RULES = {
    "IT": [
        "software", "developer", "engineer", "data", "ml", "ai", "cloud", "devops",
        "backend", "frontend", "full stack", "fullstack", "python", "java", "sql", "sap"
    ],
    "Marketing": [
        "marketing", "seo", "sea", "content", "brand", "social media",
        "performance marketing", "growth", "campaign"
    ],
    "Sales": [
        "sales", "account executive", "business development", "bd", "key account",
        "customer success", "crm", "pipeline"
    ],
    "Finance": [
        "finance", "controller", "accountant", "audit", "tax", "treasury",
        "cfo", "financial analyst"
    ],
    "HR": [
        "hr", "human resources", "recruit", "recruiter", "talent acquisition",
        "people operations", "people partner"
    ],
    "Operations": [
        "operations", "supply chain", "logistics", "procurement", "warehouse",
        "planner", "inventory", "demand planning"
    ],
    "Consulting": [
        "consultant", "advisory", "strategy", "management consulting"
    ],
    "Legal": [
        "legal", "lawyer", "counsel", "compliance", "contract"
    ]
}

SENIORITY_RULES = {
    "Intern": ["intern", "working student", "werkstudent", "trainee", "praktik"],
    "Entry": ["junior", "jr", "graduate", "entry level"],
    "Professional": ["associate", "specialist", "analyst", "engineer", "consultant"],
    "Senior": ["senior", "sr", "principal", "lead", "staff"],
    "Management": ["manager", "head", "director", "vp", "chief", "ceo", "cto", "cfo"]
}


In [9]:
def score_rules(text, rules):
    """
    Returns a dict: label -> score (#keyword hits)
    """
    t = text.lower()
    scores = {}
    for label, kws in rules.items():
        s = 0
        for kw in kws:
            if kw in t:
                s += 1
        scores[label] = s
    return scores

def predict_from_scores(scores, default="Other"):
    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]
    if best_score == 0:
        return default, 0
    return best_label, best_score

def rule_predict(text):
    dom_scores = score_rules(text, DOMAIN_RULES)
    sen_scores = score_rules(text, SENIORITY_RULES)

    dom_pred, dom_score = predict_from_scores(dom_scores, default="Other")
    sen_pred, sen_score = predict_from_scores(sen_scores, default="Other")

    return dom_pred, dom_score, sen_pred, sen_score


In [10]:
df_eval = df_gold.copy()

preds = df_eval["text"].apply(rule_predict)
df_eval["domain_pred"] = preds.apply(lambda x: x[0])
df_eval["domain_score"] = preds.apply(lambda x: x[1])
df_eval["seniority_pred"] = preds.apply(lambda x: x[2])
df_eval["seniority_score"] = preds.apply(lambda x: x[3])

print("BASELINE DOMAIN accuracy:", accuracy_score(df_eval["domain_true"], df_eval["domain_pred"]))
print("BASELINE DOMAIN macro-F1:", f1_score(df_eval["domain_true"], df_eval["domain_pred"], average="macro"))
print(classification_report(df_eval["domain_true"], df_eval["domain_pred"]))

print("BASELINE SENIORITY accuracy:", accuracy_score(df_eval["seniority_true"], df_eval["seniority_pred"]))
print("BASELINE SENIORITY macro-F1:", f1_score(df_eval["seniority_true"], df_eval["seniority_pred"], average="macro"))
print(classification_report(df_eval["seniority_true"], df_eval["seniority_pred"]))


BASELINE DOMAIN accuracy: 0.39992418498862775
BASELINE DOMAIN macro-F1: 0.12704549859710085
                        precision    recall  f1-score   support

        Administrative       0.00      0.00      0.00        84
  Business Development       0.00      0.00      0.00        78
            Consulting       0.85      0.44      0.58       195
      Customer Support       0.00      0.00      0.00        48
               Finance       0.00      0.00      0.00         0
                    HR       0.00      0.00      0.00         0
       Human Resources       0.00      0.00      0.00        70
                    IT       0.00      0.00      0.00         0
Information Technology       0.00      0.00      0.00       312
                 Legal       0.00      0.00      0.00         0
             Marketing       0.38      0.34      0.36       133
            Operations       0.00      0.00      0.00         0
                 Other       0.53      0.67      0.59      1252
    Project

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [11]:
df_pred = df_un.copy()

preds_un = df_pred["text"].apply(rule_predict)
df_pred["pred_domain_baseline"] = preds_un.apply(lambda x: x[0])
df_pred["pred_seniority_baseline"] = preds_un.apply(lambda x: x[2])

OUTFILE = "predictions_baseline_rulebased.csv"
df_pred[["idx","pred_domain_baseline","pred_seniority_baseline"]].to_csv(OUTFILE, index=False)

print("Saved:", OUTFILE)
df_pred.head()


Saved: predictions_baseline_rulebased.csv


,idx,text,status,pred_domain_baseline,pred_seniority_baseline
0,0,"position: Bookkeeper | org: Keeping The Books,...",ACTIVE,Other,Other
1,1,position: Co-Owner | org: Playful Paws | start...,ACTIVE,Other,Other
2,2,position: Logistics Officer | org: S&R service...,INACTIVE,Operations,Other
3,3,position: Truck driver/ laborer | org: ABC Sup...,INACTIVE,Other,Other
4,4,position: Fuel Driver | org: MB Railways | sta...,INACTIVE,IT,Other


In [12]:
print("Most common baseline domain predictions:")
display(df_pred["pred_domain_baseline"].value_counts().head(15))

print("\nExamples predicted as Other (domain):")
display(df_pred[df_pred["pred_domain_baseline"]=="Other"][["text","pred_domain_baseline","pred_seniority_baseline"]].head(10))


Most common baseline domain predictions:


,count
pred_domain_baseline,
Other,1102
IT,269
HR,136
Marketing,119
Consulting,101
Sales,76
Operations,36
Finance,32
Legal,15



Examples predicted as Other (domain):


,text,pred_domain_baseline,pred_seniority_baseline
0,"position: Bookkeeper | org: Keeping The Books,...",Other,Other
1,position: Co-Owner | org: Playful Paws | start...,Other,Other
3,position: Truck driver/ laborer | org: ABC Sup...,Other,Other
5,position: Food Delivery Driver | org: Sysco | ...,Other,Other
6,position: Regional driver | org: UPS Freight |...,Other,Other
11,position: Law Clerk | org: Baker McKenzie | st...,Other,Other
14,position: Regulatory Reporting Specialist | or...,Other,Professional
15,position: Event Support | org: Se2 Solutions S...,Other,Other
16,position: Customer Service Assistant | org: DC...,Other,Other
19,position: Intern | org: ESW Consulting Wruss Z...,Other,Intern
